In [1]:
import numpy as np
import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.metrics import root_mean_squared_error,mean_absolute_error,r2_score,roc_auc_score
from xgboost import XGBRegressor

In [2]:
df = pd.read_csv('AmesHousing.csv')
df.set_index('Order',inplace=True)

In [3]:
df.head()

,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,Utilities,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
Order,,,,,,,,,,,,,,,,,,,,,
1,526301100,20,RL,141.0,31770,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2010,WD,Normal,215000
2,526350040,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal,105000
3,526351010,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal,172000
4,526353030,20,RL,93.0,11160,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,244000
5,527105010,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal,189900


In [4]:
df.tail()

,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,Utilities,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
Order,,,,,,,,,,,,,,,,,,,,,
2926,923275080,80,RL,37.0,7937,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,GdPrv,NaN,0,3,2006,WD,Normal,142500
2927,923276100,20,RL,NaN,8885,Pave,NaN,IR1,Low,AllPub,...,0,NaN,MnPrv,NaN,0,6,2006,WD,Normal,131000
2928,923400125,85,RL,62.0,10441,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,MnPrv,Shed,700,7,2006,WD,Normal,132000
2929,924100070,20,RL,77.0,10010,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,4,2006,WD,Normal,170000
2930,924151050,60,RL,74.0,9627,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,11,2006,WD,Normal,188000


In [5]:
df.columns

Index(['PID', 'MS SubClass', 'MS Zoning', 'Lot Frontage', 'Lot Area', 'Street',
       'Alley', 'Lot Shape', 'Land Contour', 'Utilities', 'Lot Config',
       'Land Slope', 'Neighborhood', 'Condition 1', 'Condition 2', 'Bldg Type',
       'House Style', 'Overall Qual', 'Overall Cond', 'Year Built',
       'Year Remod/Add', 'Roof Style', 'Roof Matl', 'Exterior 1st',
       'Exterior 2nd', 'Mas Vnr Type', 'Mas Vnr Area', 'Exter Qual',
       'Exter Cond', 'Foundation', 'Bsmt Qual', 'Bsmt Cond', 'Bsmt Exposure',
       'BsmtFin Type 1', 'BsmtFin SF 1', 'BsmtFin Type 2', 'BsmtFin SF 2',
       'Bsmt Unf SF', 'Total Bsmt SF', 'Heating', 'Heating QC', 'Central Air',
       'Electrical', '1st Flr SF', '2nd Flr SF', 'Low Qual Fin SF',
       'Gr Liv Area', 'Bsmt Full Bath', 'Bsmt Half Bath', 'Full Bath',
       'Half Bath', 'Bedroom AbvGr', 'Kitchen AbvGr', 'Kitchen Qual',
       'TotRms AbvGrd', 'Functional', 'Fireplaces', 'Fireplace Qu',
       'Garage Type', 'Garage Yr Blt', 'Garage Finish'

In [ ]:
df.info()

In [ ]:
columns_object = df.select_dtypes(include='object')
columns_int = df.select_dtypes(include='int')
columns_float = df.select_dtypes(include='float')

In [ ]:
columns_object.shape[1] + columns_int.shape[1] + columns_float.shape[1] == df.shape[1]

In [ ]:
columns_object[:10]

In [ ]:
df.describe()

In [ ]:
plt.figure(figsize=(15,8))
top_corr = df.corr(numeric_only=True)['SalePrice'].sort_values(ascending=False).head(10)
sns.heatmap(    
                df[top_corr.index].corr(),
                annot=True,
                cmap = 'Reds'
    )
plt.show()

In [ ]:
missing_value_columns = []
for x in df.columns:
    if df[x].isna().sum() != 0:
        missing_value_columns.append(x)

In [ ]:
import missingno as msno

In [ ]:
msno.matrix(
                df = df[missing_value_columns], # Original dataset
                figsize=(25,14),
                fontsize = 20,
                label_rotation = 35,
                labels=True
)
plt.show()

In [ ]:
msno.heatmap(
                df = df[missing_value_columns]
)
plt.show()

In [ ]:
percentage_missing_values_in_columns = pd.DataFrame({
    'columns'             : missing_value_columns,
    'percentage_missing'  :  ((df[missing_value_columns].isna().sum()/df.shape[0]*100).values)
}).sort_values(by='percentage_missing',ascending=False)

In [ ]:
percentage_missing_values_in_columns

In [ ]:
plt.figure(figsize=(15,8))
sns.barplot(
                data = percentage_missing_values_in_columns,
                x = 'columns',
                y = 'percentage_missing',
                hue = 'columns',
                palette='magma',
                width = 0.75,
                saturation = True
)
plt.title('Missing Value percentage in the dataset')
plt.xticks(rotation=35)
plt.show()

In [ ]:
plt.figure(figsize=(10,5))
sns.histplot(
                data = df,
                x = 'SalePrice',
                color = 'black',
                kde = True,
)
plt.show()

In [ ]:
for x in columns_object.columns:
    if(len(columns_object[x].unique()) < 10):
        print(f' Column {x} : \n {columns_object[x].unique()} \n ')